# Matrix Factorization

In [1]:
!pip install pyspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9


In [18]:
#!pip install -U py4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.7
    Uninstalling py4j-0.10.9.7:
      Successfully uninstalled py4j-0.10.9.7
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyspark 3.5.1 requires py4j==0.10.9.7, but you have py4j 0.10.9.9 which is incompatible.


In [2]:
!apt-get install -y openjdk-17-jdk-headless -qq

In [3]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] += ":/usr/lib/jvm/java-17-openjdk-amd64/bin"

In [1]:
import pandas as pd
import numpy as np

## Reading Ratings Data

In [4]:
ratings_df = pd.read_csv('https://raw.githubusercontent.com/manaranjanp/MLUL2/refs/heads/main/mf/u.data', sep = '\t')

In [5]:
ratings_df

,196,242,3,881250949
0,186,302,3,891717742
1,22,377,1,878887116
2,244,51,2,880606923
3,166,346,1,886397596
4,298,474,4,884182806
...,...,...,...,...
99994,880,476,3,880175444
99995,716,204,5,879795543
99996,276,1090,1,874795795
99997,13,225,2,882399156


In [6]:
ratings_df.columns = ['userId', 'itemId', 'rating', 'timestamp']

In [7]:
ratings_df

,userId,itemId,rating,timestamp
0,186,302,3,891717742
1,22,377,1,878887116
2,244,51,2,880606923
3,166,346,1,886397596
4,298,474,4,884182806
...,...,...,...,...
99994,880,476,3,880175444
99995,716,204,5,879795543
99996,276,1090,1,874795795
99997,13,225,2,882399156


In [8]:
ratings_df.drop('timestamp', axis = 1, inplace = True)

In [9]:
len(ratings_df.userId.unique())

943

In [10]:
len(ratings_df.itemId.unique())

1682

## Reading the movies metadata

In [11]:
movies_df = pd.read_csv('https://raw.githubusercontent.com/manaranjanp/MLUL2/refs/heads/main/mf/u.item',
                        encoding = 'iso-8859-1',
                        sep = '|',
                        header = None,
                        usecols=[0, 1])

In [12]:
movies_df

,0,1
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)
...,...,...
1677,1678,Mat' i syn (1997)
1678,1679,B. Monkey (1998)
1679,1680,Sliding Doors (1998)
1680,1681,You So Crazy (1994)


In [13]:
movies_df.columns = ['itemId', 'name']

In [14]:
movies_df.head(10)

,itemId,name
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)
5,6,Shanghai Triad (Yao a yao yao dao waipo qiao) ...
6,7,Twelve Monkeys (1995)
7,8,Babe (1995)
8,9,Dead Man Walking (1995)
9,10,Richard III (1995)


### Matrix Factorization Methods

In [15]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType

def als_factorize_numpy(R, num_factors=3, reg_param=0.1, max_iter=20):
    """
    Perform matrix factorization using PySpark ALS on an incomplete NumPy matrix.

    Parameters
    ----------
    R : np.ndarray
        User-item matrix with NaN for missing entries.
    num_factors : int
        Number of latent features (rank of factorization).
    reg_param : float
        Regularization parameter (lambda).
    max_iter : int
        Number of ALS iterations.

    Returns
    -------
    W : np.ndarray
        User-feature matrix (num_users x num_factors)
    H : np.ndarray
        Item-feature matrix (num_items x num_factors)
    """

    # --- Step 1: Initialize Spark ---
    spark = SparkSession.builder \
        .appName("ALS Matrix Factorization") \
        .getOrCreate()

    ratings_df = spark.createDataFrame(R)

    # --- Step 3: Initialize ALS ---
    als = ALS(
        userCol="userId",
        itemCol="itemId",
        ratingCol="rating",
        rank=num_factors,
        regParam=reg_param,
        maxIter=max_iter,
        coldStartStrategy="drop",
        checkpointInterval=10,  # Add checkpointing
        nonnegative=True
    )

    # --- Step 4: Train ALS model ---
    model = als.fit(ratings_df)

    # --- Step 5: Extract user and item factors ---
    user_factors = model.userFactors.toPandas().sort_values("id")
    item_factors = model.itemFactors.toPandas().sort_values("id")

    # Convert list of features to NumPy arrays
    W = np.vstack(user_factors["features"].values)
    H = np.vstack(item_factors["features"].values)

    # --- Step 6: Stop Spark session ---
    # spark.stop()

    # Predict on test
    predictions = model.transform(ratings_df)

    # Evaluate MSE
    evaluator = RegressionEvaluator(
        metricName="mse",
        labelCol="rating",
        predictionCol="prediction"
    )
    mse = evaluator.evaluate(predictions)

    return W, H, mse

## Factorizing User-Movies Ratings Matrix

In [16]:
num_factors = 10
reg_param=0.01
max_iter=10

W, H, mse = als_factorize_numpy(ratings_df, num_factors, reg_param, max_iter)

print("W (User Feature Matrix):")
print(W)
print("\nH (Item Feature Matrix):")
print(H)
print("--------------------------------------------------------------")
print(f"Final MSE = {mse:.4f}")
print("--------------------------------------------------------------")

W (User Feature Matrix):
[[0.88967109 0.         0.         ... 0.67848116 1.08398497 0.71547323]
 [0.28948796 0.19277048 0.33579382 ... 0.99561566 0.04999527 1.29487503]
 [0.         0.49759299 0.         ... 0.40913537 0.53804493 0.85740513]
 ...
 [0.4139623  0.57933056 0.44223756 ... 1.54104745 0.61085093 1.29199553]
 [0.97486627 0.8556866  0.35353863 ... 0.94822901 0.64662021 1.66302192]
 [0.73241848 0.         0.29713312 ... 0.         0.14679566 1.42903793]]

H (Item Feature Matrix):
[[0.25084251 0.         0.29837418 ... 0.76462144 0.80077469 1.16342592]
 [0.49537227 0.1169948  0.01981172 ... 0.23970298 0.85944414 0.98135626]
 [2.15336227 0.62122858 0.21069056 ... 0.         0.61538249 0.51620013]
 ...
 [0.25577769 0.         0.         ... 0.         0.         0.45445979]
 [0.2759653  0.         0.13989204 ... 0.62159109 0.22644778 1.02356243]
 [0.61759561 0.13716826 0.20837277 ... 0.55283654 0.19286545 0.49066567]]
-------------------------------------------------------------

In [17]:
W.shape

(943, 10)

In [18]:
H.shape

(1682, 10)

## Finding Similarity

In [19]:
from sklearn.metrics import pairwise_distances
from scipy.spatial.distance import cosine, correlation

movies_sim = 1 - pairwise_distances( H, metric="cosine" )
movies_sim_df = pd.DataFrame( movies_sim )

In [20]:
def get_similar_movies( itemId, topN = 10 ):
    movieidx = movies_df[movies_df.itemId == itemId].index[0]
    movies_df['similarity'] = movies_sim_df.iloc[movieidx]
    top_n = movies_df.sort_values( ["similarity"], ascending = False )[0:topN]
    return top_n

In [21]:
movies_sim_df

,0,1,2,3,4,5,6,7,8,9,...,1672,1673,1674,1675,1676,1677,1678,1679,1680,1681
0,1.000000,0.835548,0.414114,0.801418,0.806882,0.403171,0.857937,0.934241,0.753424,0.683868,...,0.728110,0.807831,0.842000,0.842000,0.812328,0.744427,0.744427,0.744427,0.933794,0.812008
1,0.835548,1.000000,0.570759,0.711751,0.805409,0.421099,0.756794,0.786533,0.556954,0.650118,...,0.704107,0.659504,0.750840,0.750840,0.588379,0.593081,0.593081,0.593081,0.720614,0.647746
2,0.414114,0.570759,1.000000,0.703812,0.541105,0.098363,0.568572,0.499816,0.371509,0.599161,...,0.577418,0.611649,0.582195,0.582195,0.427773,0.507026,0.507026,0.507026,0.424545,0.609644
3,0.801418,0.711751,0.703812,1.000000,0.677644,0.514896,0.924844,0.765656,0.803781,0.709846,...,0.843002,0.799063,0.937427,0.937427,0.917372,0.854879,0.854879,0.854879,0.804799,0.937271
4,0.806882,0.805409,0.541105,0.677644,1.000000,0.314162,0.711820,0.861379,0.684596,0.693504,...,0.824820,0.866907,0.741242,0.741242,0.623685,0.622245,0.622245,0.622245,0.792755,0.715759
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1677,0.744427,0.593081,0.507026,0.854879,0.622245,0.445347,0.761327,0.688748,0.674462,0.364060,...,0.706531,0.697655,0.909314,0.909314,0.777999,1.000000,1.000000,1.000000,0.836708,0.833238
1678,0.744427,0.593081,0.507026,0.854879,0.622245,0.445347,0.761327,0.688748,0.674462,0.364060,...,0.706531,0.697655,0.909314,0.909314,0.777999,1.000000,1.000000,1.000000,0.836708,0.833238
1679,0.744427,0.593081,0.507026,0.854879,0.622245,0.445347,0.761327,0.688748,0.674462,0.364060,...,0.706531,0.697655,0.909314,0.909314,0.777999,1.000000,1.000000,1.000000,0.836708,0.833238
1680,0.933794,0.720614,0.424545,0.804799,0.792755,0.374025,0.743117,0.889127,0.760681,0.549389,...,0.782399,0.786144,0.806902,0.806902,0.787566,0.836708,0.836708,0.836708,1.000000,0.852099


## Finding Similar Movies

In [22]:
get_similar_movies(127)

,itemId,name,similarity
126,127,"Godfather, The (1972)",1.000000
186,187,"Godfather: Part II, The (1974)",0.980982
660,661,High Noon (1952),0.977031
134,135,2001: A Space Odyssey (1968),0.973151
493,494,His Girl Friday (1940),0.966979
522,523,Cool Hand Luke (1967),0.956774
176,177,"Good, The Bad and The Ugly, The (1966)",0.953982
510,511,Lawrence of Arabia (1962),0.953794
1527,1528,Nowhere (1997),0.953159
57,58,Quiz Show (1994),0.949547


In [23]:
get_similar_movies(118)

,itemId,name,similarity
117,118,Twister (1996),1.000000
81,82,Jurassic Park (1993),0.950431
120,121,Independence Day (ID4) (1996),0.948880
65,66,While You Were Sleeping (1995),0.947279
264,265,"Hunt for Red October, The (1990)",0.943039
160,161,Top Gun (1986),0.940490
1643,1644,Sudden Manhattan (1996),0.929055
814,815,One Fine Day (1996),0.928820
53,54,Outbreak (1995),0.926467
327,328,Conspiracy Theory (1997),0.924889
